# Chapter 16: Support Vector Machines & k-Nearest Neighbors
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksplorasi, memodelkan, mentransformasi kernel, dan mengevaluasi algoritma SVM dan KNN:
1. **Eksplorasi Dataset Telemetri Jaringan Sensor Nirkabel (WSN)**.
2. **Prapemrosesan & Kebutuhan Mutlak Feature Scaling**.
3. **Linear Support Vector Machine & Support Vectors**.
4. **Eksplorasi Kernel Trick (Polynomial vs RBF Kernel)**.
5. **Analisis Sensitivitas Hyperparameter C dan Gamma**.
6. **Instance-Based Learning: k-Nearest Neighbors (KNN)**.
7. **Pengaruh Nilai k dan Metrik Jarak (Euclidean vs Manhattan)**.
8. **Benchmark Komparasi Komprehensif SVM vs KNN**.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Muat dataset sensor nirkabel
data_path = Path("../code/data/wireless_sensor_network_state.csv")
df = pd.read_csv(data_path)
print(f"Dimensi Dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
print(f"Distribusi Target: {dict(df['node_status'].value_counts())}")
df.head(3)


Dimensi Dataset: 360 baris x 8 kolom
Distribusi Target: {'CRITICAL': np.int64(140), 'WARNING': np.int64(123), 'NORMAL': np.int64(97)}


## 2. Pemisahan Data & Pembuktian Efek Feature Scaling
Mengapa standarisasi fitur wajib untuk model geometris dan berbasis jarak?

In [2]:
features = ['signal_strength_dbm', 'temperature_c', 'humidity_pct', 'packet_loss_pct', 'latency_ms', 'sensor_voltage_v']
X = df[features]
y = df['node_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Uji RBF SVM tanpa scaling vs dengan StandardScaler
raw_svm = SVC(kernel='rbf', random_state=42).fit(X_train, y_train)
scaled_pipe = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf', random_state=42))]).fit(X_train, y_train)

print(f"Akurasi RBF SVM TANPA Scaling     : {accuracy_score(y_test, raw_svm.predict(X_test))*100:.2f}%")
print(f"Akurasi RBF SVM DENGAN Standarisasi: {accuracy_score(y_test, scaled_pipe.predict(X_test))*100:.2f}%")


Akurasi RBF SVM TANPA Scaling     : 58.89%
Akurasi RBF SVM DENGAN Standarisasi: 70.00%


## 3. Linear SVM & Vektor Pendukung (*Support Vectors*)
Menemukan hiperbidang pemisah margin maksimum dan mengidentifikasi sampel kritis di perbatasan margin.

In [3]:
pipe_lin = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=1.0, random_state=42))
])
pipe_lin.fit(X_train, y_train)
print(f"Akurasi Uji Linear SVM: {pipe_lin.score(X_test, y_test)*100:.2f}%")
print(f"Total Support Vectors: {sum(pipe_lin.named_steps['svm'].n_support_)} dari {len(X_train)} data latih")


Akurasi Uji Linear SVM: 62.22%
Total Support Vectors: 216 dari 270 data latih


## 4. Kernel Methods: RBF Kernel Non-Linier
Memetakan data ke ruang berdimensi tinggi implisit menggunakan trik kernel Gaussian RBF.

In [4]:
pipe_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42))
])
pipe_rbf.fit(X_train, y_train)
print(f"Akurasi Uji RBF SVM (C=1.0, gamma='scale'): {pipe_rbf.score(X_test, y_test)*100:.2f}%")
print(f"Total Support Vectors RBF: {sum(pipe_rbf.named_steps['svm'].n_support_)}")


Akurasi Uji RBF SVM (C=1.0, gamma='scale'): 70.00%
Total Support Vectors RBF: 212


## 5. K-Nearest Neighbors Classifier (KNN)
Instance-based learning menggunakan jarak Euclidean dengan $k=5$ tetangga terdekat.

In [5]:
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, weights='uniform'))
])
pipe_knn.fit(X_train, y_train)
print(f"Akurasi Uji KNN (k=5): {pipe_knn.score(X_test, y_test)*100:.2f}%")


Akurasi Uji KNN (k=5): 57.78%


## 6. Laporan Evaluasi Klasifikasi Model Terbaik (RBF SVM)

In [6]:
y_pred_best = pipe_rbf.predict(X_test)
print(classification_report(y_test, y_pred_best, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best, labels=pipe_rbf.classes_))


              precision    recall  f1-score   support

    CRITICAL     0.6471    0.6286    0.6377        35
      NORMAL     0.7333    0.9167    0.8148        24
     WARNING     0.7308    0.6129    0.6667        31

    accuracy                         0.7000        90
   macro avg     0.7037    0.7194    0.7064        90
weighted avg     0.6989    0.7000    0.6949        90

Confusion Matrix:
[[22  7  6]
 [ 1 22  1]
 [11  1 19]]
